# PitchsideIQ: Feature Engineering & Preprocessing Pipeline
### Engineering Dynamic Pre-Match Predictive Signals Without Future Data Leakage

**Project Goal:** Convert static raw match rows into rich dynamic features reflecting team strength, recent momentum, fatigue, and market sentiment, strictly respecting temporal integrity.

#### Core Feature Engineering Modules:
1. **Dynamic Elo Rating Engine:** Simulation with margin-of-victory and home field advantage multiplier (+65 Elo).
2. **Opponent-Adjusted Form:** Rolling 5-match form scaled by the quality of opposing teams (`opp_elo / 1500.0`).
3. **Total Shot Ratio (TSR) Rolling Momentum:** Shot generation dominance `Shots_For / (Shots_For + Shots_Against)`.
4. **Fatigue & Rest Turnaround:** Days since previous fixture, capturing congestion from midweek continental matches.
5. **Market Probabilities & Margin Normalization:** Extracting consensus implied probabilities and bookmaker vigorish.
6. **Leak-Free Guard:** Updating team states *after* each match is played, ensuring zero look-ahead bias.

In [ ]:
import pandas as pd
import numpy as np
import math
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print("Preprocessing pipeline initialized!")

## 1. Load Cleaned Base Match Records

In [ ]:
candidate_paths = [
    Path("data/cleaned_base_matches.csv"),
    Path("cleaned_base_matches.csv"),
    Path("../data/cleaned_base_matches.csv"),
    Path("raw_data.csv"),
    Path("../raw_data.csv")
]
base_path = next((p for p in candidate_paths if p.exists()), Path("raw_data.csv"))
print(f"Loading match data from: {base_path.resolve()}")

df = pd.read_csv(base_path, low_memory=False, parse_dates=["MatchDate"] if "MatchDate" in pd.read_csv(base_path, nrows=1).columns else None)
if "MatchDate" not in df.columns and "Date" in df.columns:
    df["MatchDate"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
else:
    df["MatchDate"] = pd.to_datetime(df["MatchDate"], dayfirst=True, errors="coerce")

df = df.dropna(subset=["MatchDate", "HomeTeam", "AwayTeam", "FTHome" if "FTHome" in df.columns else "FTHG"])
df = df.sort_values("MatchDate").reset_index(drop=True)

# Standardize columns if raw
if "FTHome" not in df.columns and "FTHG" in df.columns:
    df["FTHome"] = df["FTHG"].astype(int)
    df["FTAway"] = df["FTAG"].astype(int)
    df["FTResult"] = df["FTR"].astype(str).str.strip().str.upper()

print(f"Loaded {len(df):,} matches sorted chronologically.")

## 2. Dynamic 20-Year Elo Rating Engine
We simulate match results chronologically.
- Base Elo: $1500.0$
- Home Field Advantage: $+65.0$ Elo points
- Expected Score Formula:
$$E_H = \frac{1}{1 + 10^{-(Elo_H + 65 - Elo_A)/400}}$$
- Margin-of-Victory Scaling: Larger goal margins yield greater rating adjustments:
$$\text{Multiplier} = \ln(\Delta \text{Goals} + 1) \times \text{ScaleFactor}$$

In [ ]:
BASE_ELO = 1500.0
HOME_ADVANTAGE = 65.0
K = 24.0

elo_ratings = {}
team_history = {}

home_elos, away_elos, elo_diffs = [], [], []
adj_form_h, adj_form_a = [], []
tsr_h, tsr_a = [], []
rest_h, rest_a = [], []

for idx, row in df.iterrows():
    h_team = row["HomeTeam"]
    a_team = row["AwayTeam"]
    m_date = row["MatchDate"]

    h_elo = elo_ratings.get(h_team, BASE_ELO)
    a_elo = elo_ratings.get(a_team, BASE_ELO)
    h_elos_eff = h_elo + HOME_ADVANTAGE
    diff = h_elos_eff - a_elo

    home_elos.append(h_elo)
    away_elos.append(a_elo)
    elo_diffs.append(diff)

    # 1. Rolling Home Team Stats
    h_hist = team_history.get(h_team, [])
    if h_hist:
        last_date = h_hist[-1]["date"]
        days_rest = (m_date - last_date).days
        recent_5 = h_hist[-5:]
        f_adj = sum(m["pts"] * (m["opp_elo"] / 1500.0) for m in recent_5) / max(len(recent_5), 1)
        sf = sum(m["shots_for"] for m in recent_5)
        sa = sum(m["shots_against"] for m in recent_5)
        tsr = sf / max(sf + sa, 1)
    else:
        days_rest = 7
        f_adj = 1.3
        tsr = 0.50
    rest_h.append(min(max(days_rest, 1), 28))
    adj_form_h.append(round(f_adj, 3))
    tsr_h.append(round(tsr, 3))

    # 2. Rolling Away Team Stats
    a_hist = team_history.get(a_team, [])
    if a_hist:
        last_date = a_hist[-1]["date"]
        days_rest = (m_date - last_date).days
        recent_5 = a_hist[-5:]
        f_adj = sum(m["pts"] * (m["opp_elo"] / 1500.0) for m in recent_5) / max(len(recent_5), 1)
        sf = sum(m["shots_for"] for m in recent_5)
        sa = sum(m["shots_against"] for m in recent_5)
        tsr = sf / max(sf + sa, 1)
    else:
        days_rest = 7
        f_adj = 1.3
        tsr = 0.50
    rest_a.append(min(max(days_rest, 1), 28))
    adj_form_a.append(round(f_adj, 3))
    tsr_a.append(round(tsr, 3))

    # POST-MATCH UPDATE
    res = row["FTResult"]
    actual_h = 1.0 if res == "H" else (0.5 if res == "D" else 0.0)
    expected_h = 1.0 / (1.0 + 10.0 ** (-diff / 400.0))
    goal_diff = abs(row["FTHome"] - row["FTAway"])
    mov_mult = math.log(max(goal_diff, 1) + 1.0) * (2.2 / ((h_elo - a_elo) * 0.001 + 2.2) if actual_h == 1 else 1.0)
    mov_mult = max(mov_mult, 1.0)

    delta = K * mov_mult * (actual_h - expected_h)
    elo_ratings[h_team] = h_elo + delta
    elo_ratings[a_team] = a_elo - delta

    h_pts = 3 if res == "H" else (1 if res == "D" else 0)
    a_pts = 3 if res == "A" else (1 if res == "D" else 0)
    hs = row.get("HomeShots", 12) if pd.notna(row.get("HomeShots")) else 12
    as_ = row.get("AwayShots", 10) if pd.notna(row.get("AwayShots")) else 10

    if h_team not in team_history:
        team_history[h_team] = []
    team_history[h_team].append({"date": m_date, "pts": h_pts, "opp_elo": a_elo, "shots_for": hs, "shots_against": as_})

    if a_team not in team_history:
        team_history[a_team] = []
    team_history[a_team].append({"date": m_date, "pts": a_pts, "opp_elo": h_elo, "shots_for": as_, "shots_against": hs})

df["HomeElo"] = home_elos
df["AwayElo"] = away_elos
df["EloDiff"] = elo_diffs
df["AdjFormHome"] = adj_form_h
df["AdjFormAway"] = adj_form_a
df["AdjFormDiff"] = np.array(adj_form_h) - np.array(adj_form_a)
df["TSRHome"] = tsr_h
df["TSRAway"] = tsr_a
df["TSRDiff"] = np.array(tsr_h) - np.array(tsr_a)
df["RestHome"] = rest_h
df["RestAway"] = rest_a
df["RestDiff"] = np.array(rest_h) - np.array(rest_a)

print("Dynamic Elo, Form, TSR, and Rest features successfully engineered!")

## 3. Market Implied Probabilities & Margin Extraction
Converting raw decimal odds into normalized probability signals:
$$P_H = \frac{1}{\text{Odd}_H}, \quad P_D = \frac{1}{\text{Odd}_D}, \quad P_A = \frac{1}{\text{Odd}_A}$$
$$\text{Overround Margin} = (P_H + P_D + P_A) - 1.0$$

In [ ]:
for side, code in [("Home", "H"), ("Draw", "D"), ("Away", "A")]:
    out_col = f"Odd{side}"
    if out_col not in df.columns:
        candidates = [f"Avg{code}", f"B365{code}", f"WH{code}"]
        extracted = None
        for c in candidates:
            if c in df.columns:
                val = pd.to_numeric(df[c], errors="coerce")
                extracted = val if extracted is None else extracted.fillna(val)
        df[out_col] = extracted

for side in ["Home", "Draw", "Away"]:
    odd_col = f"Odd{side}"
    df[f"ImpProb{side}"] = 1.0 / df[odd_col].where(df[odd_col] > 1.0)

df["OddsMargin"] = df["ImpProbHome"] + df["ImpProbDraw"] + df["ImpProbAway"] - 1.0
df["Month"] = df["MatchDate"].dt.month.astype(float)
df["Year"] = df["MatchDate"].dt.year.astype(int)

print("Market probability features extracted.")

## 4. Modeling Targets Formulation
1. `outcome_1x2`: 3-class target ($0 = \text{Home Win}, 1 = \text{Draw}, 2 = \text{Away Win}$).
2. `fav_avoids_defeat`: Binary indicator ($1$ if the market favorite wins or draws, $0$ if upset).

In [ ]:
df["outcome_1x2"] = df["FTResult"].map({"H": 0, "D": 1, "A": 2})

# Determine market favorite
df["home_is_fav"] = df["OddHome"] <= df["OddAway"]
df["fav_avoids_defeat"] = np.where(
    df["home_is_fav"],
    df["FTResult"].isin(["H", "D"]).astype(int),
    df["FTResult"].isin(["A", "D"]).astype(int)
)

print("Target distributions:")
print("1X2 Outcome:", df["outcome_1x2"].value_counts().to_dict())
print("Favorite Avoids Defeat Rate:", round(df["fav_avoids_defeat"].mean() * 100, 2), "%")

## 5. Export Final Processed Dataset
Saving `processed_data.csv` for training machine learning models.

In [ ]:
out_candidates = [Path("processed_data.csv"), Path("../processed_data.csv")]
out_path = out_candidates[0] if Path("data").exists() else out_candidates[1]

df.to_csv(out_path, index=False)
print(f"Exported clean, leak-free processed dataset to: {out_path.resolve()}")
print(f"Total Rows: {len(df):,}, Total Engineered Columns: {df.shape[1]}")